# RQ3 — SHAP Meta-Explanation

**Research Question**: Which dataset properties most influence which clustering method gets recommended?

Uses `shap.TreeExplainer` directly on the saved ExtraTrees classifier. This is the correct
approach — `TreeExplainer` is exact for tree-based models and requires no background
summary or sampling.

> Previous runs used `KernelExplainer` on a k=5 kNN as a workaround for a `KeyError: 'scale'`
> bug. This notebook fixes that by accessing `pipe.named_steps['clf']` (the ExtraTrees
> estimator) and `pipe.named_steps['impute']` (the only preprocessing step), passing
> imputed-but-not-scaled data to `TreeExplainer`.

**Outputs**: `outputs/figures/shap_*.png`, `data/meta_table/shap_values.csv`

In [1]:
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
from meta_learner import LSE_COLS, META_COLS

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

METHOD_COLORS = {
    'kmeans'   : '#4C72B0',
    'dbscan'   : '#DD8452',
    'agg'      : '#55A868',
    'gmm'      : '#C44E52',
    'autoenc'  : '#8172B2',
    'dictlearn': '#937860',
}
print(f'shap {shap.__version__}')

shap 0.51.0


In [2]:
df = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df = df.dropna(subset=['best_method']).reset_index(drop=True)

feat_cols = [c for c in df.columns if c not in META_COLS + LSE_COLS]
X_raw = df[feat_cols].values.astype(float)
y     = df['best_method'].values

print(f'Datasets: {len(df)},  Features: {len(feat_cols)}')
print(f'Features: {feat_cols}')

Datasets: 94,  Features: 20
Features: ['n_instances', 'n_features', 'n_classes', 'skewness_mean', 'kurtosis_mean', 'mean_abs_pearson', 'hopkins', 'intrinsic_dim_ratio', 'pca_var_pc1', 'pca_top3_var', 'pca_entropy', 'pairwise_dist_mean', 'pairwise_dist_cv', 'pairwise_dist_p90', 'knn5_dist_mean', 'knn5_dist_cv', 'knn5_dist_p90', 'feature_sparsity', 'high_corr_frac', 'corr_dispersion']


In [3]:
with open(os.path.join(MODELS_DIR, 'meta_clf_optA.pkl'), 'rb') as f:
    saved = pickle.load(f)

pipe      = saved['pipeline']
feat_cols = saved['feature_cols']  # use exactly the features the model was trained on

# The ExtraTrees pipeline: impute → clf (no scaling step)
# Impute raw X, then pass directly to TreeExplainer — do NOT scale
imputer   = pipe.named_steps['impute']
et_clf    = pipe.named_steps['clf']   # ExtraTreesClassifier

X_raw_aligned = df[feat_cols].values.astype(float)
X_imputed     = imputer.transform(X_raw_aligned)

print(f'X_imputed shape : {X_imputed.shape}')
print(f'Classes         : {et_clf.classes_}')

X_imputed shape : (94, 20)
Classes         : ['agg' 'autoenc' 'dbscan' 'dictlearn' 'gmm' 'kmeans']


In [4]:
# TreeExplainer is exact for tree-based models — no sampling, no background needed
explainer  = shap.TreeExplainer(et_clf)
shap_values = explainer.shap_values(X_imputed)  # list of (n_samples, n_features) per class

classes = list(et_clf.classes_)
n, p, k = len(X_imputed), len(feat_cols), len(classes)

shap_arr = np.array(shap_values)  # shape: (k, n, p) or (n, p, k)
if shap_arr.shape == (n, p, k):
    shap_arr = shap_arr.transpose(2, 0, 1)
elif shap_arr.shape != (k, n, p):
    raise ValueError(f'Unexpected shap_arr shape: {shap_arr.shape}')

print(f'shap_arr shape (classes, datasets, features): {shap_arr.shape}')

shap_arr shape (classes, datasets, features): (6, 94, 20)


In [5]:
rows = []
for ci, cls in enumerate(classes):
    for si in range(n):
        row = {'class': cls,
               'dataset_id': int(df['dataset_id'].iloc[si]),
               'true_method': y[si]}
        for fi, fn in enumerate(feat_cols):
            row[fn] = float(shap_arr[ci, si, fi])
        rows.append(row)

shap_df = pd.DataFrame(rows)
shap_path = os.path.join(META_DIR, 'shap_values.csv')
shap_df.to_csv(shap_path, index=False)
print(f'Saved → {shap_path}  shape={shap_df.shape}')

Saved → c:\MLResearch\data\meta_table\shap_values.csv  shape=(564, 23)


In [6]:
global_importance = np.abs(shap_arr).mean(axis=(0, 1))  # (n_features,)
imp_df = pd.DataFrame({'feature': feat_cols, 'mean_abs_shap': global_importance})
imp_df = imp_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('=== Global feature importance (ExtraTrees, exact SHAP) ===')
print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df['feature'][::-1], imp_df['mean_abs_shap'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global meta-feature importance (ExtraTrees TreeExplainer)')
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_global_importance.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

=== Global feature importance (ExtraTrees, exact SHAP) ===
            feature  mean_abs_shap
      knn5_dist_p90       0.019484
       pca_top3_var       0.018859
   feature_sparsity       0.018306
   pairwise_dist_cv       0.015197
    corr_dispersion       0.014092
        pca_var_pc1       0.014032
          n_classes       0.013150
       knn5_dist_cv       0.012750
  pairwise_dist_p90       0.011360
     knn5_dist_mean       0.010798
        n_instances       0.009884
        pca_entropy       0.008963
         n_features       0.008515
 pairwise_dist_mean       0.007403
intrinsic_dim_ratio       0.007044
      skewness_mean       0.006633
     high_corr_frac       0.006258
      kurtosis_mean       0.006206
   mean_abs_pearson       0.005311
            hopkins       0.001522
Saved → c:\MLResearch\outputs\figures\shap_global_importance.png


In [7]:
shap_mean_classes = shap_arr.mean(axis=0)  # (n, p)

shap_expl = shap.Explanation(
    values=shap_mean_classes,
    base_values=np.zeros(n),
    data=X_imputed,
    feature_names=feat_cols,
)
fig, ax = plt.subplots(figsize=(9, 7))
shap.plots.beeswarm(shap_expl, max_display=min(18, len(feat_cols)), show=False)
plt.title('SHAP beeswarm — mean across all methods\n(each dot = one dataset, ExtraTrees)')
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_beeswarm_global.png')
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close()
print(f'Saved → {path}')

Saved → c:\MLResearch\outputs\figures\shap_beeswarm_global.png


In [8]:
n_cols = min(3, len(classes))
n_rows = (len(classes) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.ravel()
per_class_top = {}

for ci, cls in enumerate(classes):
    sv       = shap_arr[ci]         # (n, p)
    mean_abs = np.abs(sv).mean(0)   # (p,)
    order    = np.argsort(mean_abs)
    top8     = order[-8:]
    top_feats = [feat_cols[i] for i in top8]
    top_vals  = mean_abs[top8]
    per_class_top[cls] = list(zip(top_feats[::-1], top_vals[::-1]))

    ax = axes[ci]
    ax.barh(top_feats, top_vals, color=METHOD_COLORS.get(cls, 'steelblue'), alpha=0.85)
    ax.set_title(f'→ {cls}  (n_best={int((y==cls).sum())})', fontweight='bold')
    ax.set_xlabel('Mean |SHAP|')
    ax.tick_params(axis='y', labelsize=8)

for ci in range(len(classes), len(axes)):
    axes[ci].set_visible(False)

plt.suptitle('Per-method feature importance (ExtraTrees)', fontsize=11, y=1.01)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_per_method.png')
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close()
print(f'Saved → {path}')

print('\n=== Top-3 features per method ===')
for cls, pairs in per_class_top.items():
    top3 = ', '.join(f'{f} ({v:.4f})' for f, v in pairs[:3])
    print(f'  {cls:12s}: {top3}')

Saved → c:\MLResearch\outputs\figures\shap_per_method.png

=== Top-3 features per method ===
  agg         : knn5_dist_p90 (0.0248), feature_sparsity (0.0156), pairwise_dist_p90 (0.0132)
  autoenc     : feature_sparsity (0.0199), knn5_dist_p90 (0.0167), corr_dispersion (0.0141)
  dbscan      : pca_top3_var (0.0316), pca_var_pc1 (0.0283), feature_sparsity (0.0275)
  dictlearn   : n_classes (0.0223), knn5_dist_p90 (0.0185), n_instances (0.0166)
  gmm         : knn5_dist_p90 (0.0283), pca_top3_var (0.0205), knn5_dist_mean (0.0184)
  kmeans      : pca_top3_var (0.0228), knn5_dist_p90 (0.0190), feature_sparsity (0.0178)


In [9]:
heat = np.zeros((len(feat_cols), len(classes)))
for ci, cls in enumerate(classes):
    heat[:, ci] = shap_arr[ci].mean(axis=0)

heat_df = pd.DataFrame(heat, index=feat_cols, columns=classes)
heat_df = heat_df.loc[imp_df['feature'].tolist()]

fig, ax = plt.subplots(figsize=(10, max(6, len(feat_cols) * 0.4)))
vmax = np.abs(heat).max()
im   = ax.imshow(heat_df.values, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(im, ax=ax, label='Mean SHAP (positive = toward method)')
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(feat_cols)))
ax.set_yticklabels(imp_df['feature'].tolist(), fontsize=8)
ax.set_title('Mean SHAP: which features steer toward each method (ExtraTrees)', fontsize=11)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_heatmap.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

Saved → c:\MLResearch\outputs\figures\shap_heatmap.png


In [10]:
df['lse_std'] = df[LSE_COLS].std(axis=1)
lse_corr = df[feat_cols + ['lse_std']].corr()['lse_std'].drop('lse_std').abs()

summary = imp_df.copy()
summary['lse_std_corr'] = summary['feature'].map(lse_corr)
summary['rank_shap']    = range(1, len(summary)+1)
summary['rank_corr']    = summary['lse_std_corr'].rank(ascending=False).astype(int)

print('=== SHAP rank vs LSE-variance correlation rank ===')
print(summary[['feature','mean_abs_shap','lse_std_corr','rank_shap','rank_corr']]
      .round(4).to_string(index=False))

thresh = min(6, len(summary))
agreed = summary[(summary['rank_shap'] <= thresh) & (summary['rank_corr'] <= thresh)]
print(f'\nTop features agreed by both rankings:')
print('  ' + ', '.join(agreed['feature'].tolist()))

print('\n=== Summary ===')
print(f"  Top feature    : {imp_df['feature'].iloc[0]}")
print(f"  Top 3          : {', '.join(imp_df['feature'].head(3).tolist())}")
print()
print('  Per-method key driver:')
for cls, pairs in per_class_top.items():
    print(f'    {cls:12s} → {pairs[0][0]} ({pairs[0][1]:.4f})')
print()
print('Phase complete. Ready for 07_showcase_eval.ipynb')

=== SHAP rank vs LSE-variance correlation rank ===
            feature  mean_abs_shap  lse_std_corr  rank_shap  rank_corr
      knn5_dist_p90         0.0195        0.1632          1         10
       pca_top3_var         0.0189        0.1241          2         14
   feature_sparsity         0.0183        0.2074          3          6
   pairwise_dist_cv         0.0152        0.1945          4          7
    corr_dispersion         0.0141        0.0372          5         19
        pca_var_pc1         0.0140        0.0204          6         20
          n_classes         0.0132        0.2321          7          4
       knn5_dist_cv         0.0128        0.2154          8          5
  pairwise_dist_p90         0.0114        0.1923          9          8
     knn5_dist_mean         0.0108        0.1295         10         13
        n_instances         0.0099        0.0753         11         17
        pca_entropy         0.0090        0.0573         12         18
         n_features       